# 04 · Multi-step pipeline: company brochure

Input: company name + URL. Output: a markdown brochure for a chosen audience.
A landing page alone is thin context, so the work is split across two model calls:

```
landing page ──► same-site links ──► [LLM: select pages, JSON] ──► fetch pages ──► [LLM: write brochure, streamed]
```

Production version: [`projects/brochure.py`](projects/brochure.py).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
from urllib.parse import urlparse

from toolkit import chat, fetch_page, stream
from toolkit.notebook import show_stream

PROVIDER = "ollama"
URL = "https://huggingface.co"

## 1. Link collection

Deterministic filtering (same host, no mailto/fragments) happens in code before the model sees anything:
cheaper, faster, and removes a class of errors.

In [ ]:
landing = fetch_page(URL)
host = urlparse(landing.url).netloc

candidates = [link for link in landing.links if urlparse(link).netloc == host]
print(f"{len(landing.links)} links, {len(candidates)} same-site")
candidates[:15]

## 2. Page selection (structured output)

JSON mode (`as_json=True` → `response_format={"type": "json_object"}`) plus a one-shot example of the
expected schema. Small local models follow the schema less reliably than frontier models.

In [ ]:
LINK_SYSTEM = """
You pick the pages of a company website that would help write a brochure:
about/company, products, customers, careers/culture. Ignore legal, login and social pages.
Return at most 4 links as JSON, exactly like this example:

{"links": [{"type": "about page", "url": "https://company.com/about"},
           {"type": "careers page", "url": "https://company.com/careers"}]}
"""

picked = chat(
    [
        {"role": "system", "content": LINK_SYSTEM},
        {"role": "user", "content": f"Links found on {URL}:\n" + "\n".join(candidates)},
    ],
    provider=PROVIDER,
    as_json=True,
)
picked

## 3. Context assembly

In [ ]:
def build_material(landing, picked_links, per_page=1_500):
    parts = [f"## Landing page\n{landing.as_prompt(2_500)}"]
    for link in picked_links:
        try:
            parts.append(f"## {link['type']}\n{fetch_page(link['url']).as_prompt(per_page)}")
        except Exception as exc:
            print("skipped", link["url"], "-", exc)
    return "\n\n".join(parts)


material = build_material(landing, picked["links"])
print(f"{len(material):,} chars of context")

## 4. Generation (streamed)

In [ ]:
def brochure_messages(company, material, tone="professional", audience="prospective customers"):
    system = (
        f"You write short company brochures in markdown (no code fences) for {audience}. "
        f"Tone: {tone}. Cover what they do, who they serve, culture and careers if mentioned. "
        "Never invent facts that aren't in the material."
    )
    return [
        {"role": "system", "content": system},
        {"role": "user", "content": f"Company: {company}\n\n{material}"},
    ]


text = show_stream(stream(brochure_messages("Hugging Face", material), provider=PROVIDER))

Audience and tone are system-prompt parameters; the gathered context is reused as-is:

In [ ]:
_ = show_stream(
    stream(
        brochure_messages("Hugging Face", material, tone="witty", audience="potential recruits"),
        provider=PROVIDER,
    )
)

## Notes

- llama3.2 occasionally invents placeholder links (`[careers page](link)`) despite the instruction;
  grounding instructions are weaker on small models.
- Next iterations: per-step model choice (cheap model for selection, stronger for writing),
  caching fetched pages, and a translation step chained after generation.